In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv('C:\\Users\\Abdullah\\Desktop\\Code\\Sentiment Based Stock\\Sentiment Based Stocks\\results\\final_dataset_labeled.csv')
df['hour'] = pd.to_datetime(df['hour'])

sentiment_cols = ['mean_finbert', 'mean_bullish', 'mean_bearish', 'mean_strength', 'post_count']
TARGET = 'label_3h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

In [3]:
scaler = StandardScaler()
train_X = scaler.fit_transform(train[sentiment_cols])
test_X  = scaler.transform(test[sentiment_cols])
train_y = train[TARGET].values
test_y  = test[TARGET].values

train_X = torch.tensor(train_X, dtype=torch.float32)
test_X  = torch.tensor(test_X,  dtype=torch.float32)
train_y = torch.tensor(train_y, dtype=torch.float32)
test_y  = torch.tensor(test_y,  dtype=torch.float32)

train_ds = torch.utils.data.TensorDataset(train_X, train_y)
test_ds  = torch.utils.data.TensorDataset(test_X,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [6]:
class SentimentClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = SentimentClassifier(input_size=len(sentiment_cols)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs   = model(X_batch).cpu().numpy()
        probs   = np.atleast_1d(probs)  # fix 0-d array issue
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

Epoch 5/30 — Loss: 0.6941
Epoch 10/30 — Loss: 0.6918
Epoch 15/30 — Loss: 0.6917
Epoch 20/30 — Loss: 0.6924
Epoch 25/30 — Loss: 0.6919
Epoch 30/30 — Loss: 0.6924


In [7]:
print("\n── Sentiment-only Results (3h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Sentiment-only Results (3h) ──
Accuracy  : 0.5190
Precision : 0.5162
Recall    : 0.9067
F1        : 0.6579
AUC       : 0.4938


In [8]:
TARGET = 'label_6h'

train = df[df['hour'] < '2025-07-01']
test  = df[df['hour'] >= '2025-07-01']

scaler = StandardScaler()
train_X = scaler.fit_transform(train[sentiment_cols])
test_X  = scaler.transform(test[sentiment_cols])
train_y = train[TARGET].values
test_y  = test[TARGET].values

train_X = torch.tensor(train_X, dtype=torch.float32)
test_X  = torch.tensor(test_X,  dtype=torch.float32)
train_y = torch.tensor(train_y, dtype=torch.float32)
test_y  = torch.tensor(test_y,  dtype=torch.float32)

train_ds = torch.utils.data.TensorDataset(train_X, train_y)
test_ds  = torch.utils.data.TensorDataset(test_X,  test_y)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [9]:
class SentimentClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = SentimentClassifier(input_size=len(sentiment_cols)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        probs   = model(X_batch).cpu().numpy()
        probs   = np.atleast_1d(probs)
        preds   = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

Epoch 5/30 — Loss: 0.6955
Epoch 10/30 — Loss: 0.6947
Epoch 15/30 — Loss: 0.6935
Epoch 20/30 — Loss: 0.6939
Epoch 25/30 — Loss: 0.6934
Epoch 30/30 — Loss: 0.6931


In [11]:
print("\n── Sentiment-only Results (6h) ──")
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1        : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC       : {roc_auc_score(all_labels, all_probs):.4f}")


── Sentiment-only Results (6h) ──
Accuracy  : 0.5123
Precision : 0.5271
Recall    : 0.6501
F1        : 0.5822
AUC       : 0.5170
